# Maple — Dataset EDA & Model Comparison
Explore the sample data and compare recommendation model outputs interactively.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

plt.rcParams['figure.figsize'] = (12, 4)
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False

## 1. Load Data

In [ ]:
interactions = pd.read_csv('../data/sample/interactions.csv', parse_dates=['timestamp'])
products     = pd.read_csv('../data/sample/products.csv')
users        = pd.read_csv('../data/sample/users.csv')

print(f'Interactions : {len(interactions):,}')
print(f'Products     : {len(products):,}')
print(f'Users        : {len(users):,}')
interactions.head()

## 2. Dataset Overview

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# Interaction type breakdown
counts = interactions['interaction_type'].value_counts()
axes[0].bar(counts.index, counts.values, color='steelblue')
axes[0].set_title('Interaction Types')
axes[0].set_ylabel('Count')

# Interactions over time
interactions.set_index('timestamp').resample('W').size().plot(ax=axes[1], color='steelblue')
axes[1].set_title('Interactions per Week')
axes[1].set_ylabel('Count')
axes[1].set_xlabel('')

# Category distribution
cat_counts = products['category'].value_counts()
axes[2].barh(cat_counts.index, cat_counts.values, color='steelblue')
axes[2].set_title('Products by Category')
axes[2].set_xlabel('Count')

plt.tight_layout()
plt.show()

## 3. User & Product Activity (Power-law distributions)

In [ ]:
user_activity    = interactions.groupby('user_id').size().sort_values(ascending=False)
product_activity = interactions.groupby('product_id').size().sort_values(ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].hist(user_activity.values, bins=50, color='steelblue', edgecolor='white')
axes[0].set_title('User Activity Distribution')
axes[0].set_xlabel('Interactions per user')
axes[0].set_ylabel('# Users')

axes[1].plot(product_activity.values, color='steelblue')
axes[1].set_title('Product Popularity (sorted)')
axes[1].set_xlabel('Product rank')
axes[1].set_ylabel('Interactions')

plt.tight_layout()
plt.show()

print(f'Top 10% of users  generate {user_activity.head(len(user_activity)//10).sum() / user_activity.sum():.1%} of interactions')
print(f'Top 10% of products get     {product_activity.head(len(product_activity)//10).sum() / product_activity.sum():.1%} of interactions')

## 4. Interaction Matrix Sparsity

In [ ]:
from src.data.loader import DataLoader

loader = DataLoader()
loader.load_interactions('../data/sample/interactions.csv')
matrix = loader.get_interaction_matrix(weighted=True)

n_users, n_items = matrix.shape
density = matrix.nnz / (n_users * n_items)

print(f'Matrix shape : {n_users} users × {n_items} products')
print(f'Non-zero     : {matrix.nnz:,}')
print(f'Density      : {density:.3%}  (sparsity: {1-density:.3%})')

# Visualise a 100x100 slice of the matrix
fig, ax = plt.subplots(figsize=(8, 6))
slice_ = matrix[:100, :100].toarray()
im = ax.imshow(slice_, aspect='auto', cmap='Blues', interpolation='nearest')
ax.set_title('Interaction matrix slice (first 100 users × 100 products)\nBlue = interaction exists')
ax.set_xlabel('Product index')
ax.set_ylabel('User index')
plt.colorbar(im, ax=ax, label='Weighted interaction value')
plt.tight_layout()
plt.show()

## 5. Train Models

In [ ]:
from src.models.popularity import PopularityRecommender
from src.models.collaborative import ItemKNNRecommender, UserKNNRecommender, ALSRecommender
from src.evaluation.metrics import evaluate_model

train_df, test_df = loader.train_test_split(test_ratio=0.2, by_time=True)

train_loader = DataLoader()
train_loader.load_interactions(df=train_df)
train_matrix = train_loader.get_interaction_matrix(weighted=True)

models = {}

print('Training Popularity...')
models['popularity'] = PopularityRecommender().fit(train_matrix)

print('Training Item-KNN...')
models['item_knn'] = ItemKNNRecommender(k=50).fit(train_matrix)

print('Training User-KNN...')
models['user_knn'] = UserKNNRecommender(k=50).fit(train_matrix)

try:
    print('Training ALS...')
    models['als'] = ALSRecommender(factors=64, iterations=15).fit(train_matrix)
except ImportError:
    print('  ALS skipped (pip install implicit)')

print('Done.')

## 6. Model Evaluation

In [ ]:
# Build test set
test_interactions = {}
for _, row in test_df.iterrows():
    if row['user_id'] in train_loader.user_id_to_idx and row['product_id'] in train_loader.product_id_to_idx:
        u = train_loader.user_id_to_idx[row['user_id']]
        i = train_loader.product_id_to_idx[row['product_id']]
        test_interactions.setdefault(u, set()).add(i)

print(f'Test users: {len(test_interactions)}')

# Evaluate all models
K_VALUES = [5, 10, 20]
all_results = {}

for name, model in models.items():
    print(f'Evaluating {name}...')
    all_results[name] = evaluate_model(
        model, test_interactions, k_values=K_VALUES, n_items=train_loader.n_products
    )

In [ ]:
# Build summary table
rows = []
for model_name, results in all_results.items():
    for k in K_VALUES:
        k_key = f'@{k}'
        row = {'model': model_name, 'k': k}
        for metric in ['precision', 'recall', 'ndcg', 'hit_rate']:
            key = f'{metric}{k_key}'
            row[metric] = results.get(key, float('nan'))
        rows.append(row)

summary = pd.DataFrame(rows)
summary.style.format({'precision': '{:.4f}', 'recall': '{:.4f}', 'ndcg': '{:.4f}', 'hit_rate': '{:.4f}'})\
       .background_gradient(subset=['ndcg'], cmap='Blues')

## 7. Metric Comparison Charts

In [ ]:
metrics = ['precision', 'recall', 'ndcg', 'hit_rate']
k_fixed = 10  # show @10

subset = summary[summary['k'] == k_fixed].set_index('model')

fig, axes = plt.subplots(1, len(metrics), figsize=(16, 4))
colors = plt.cm.Set2.colors

for ax, metric in zip(axes, metrics):
    vals = subset[metric]
    bars = ax.bar(vals.index, vals.values, color=colors[:len(vals)])
    ax.set_title(f'{metric.upper()} @ {k_fixed}')
    ax.set_ylim(0, max(vals.values) * 1.3)
    ax.tick_params(axis='x', rotation=20)
    for bar, v in zip(bars, vals.values):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.001,
                f'{v:.4f}', ha='center', va='bottom', fontsize=9)

plt.suptitle(f'Model Comparison at K={k_fixed}', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# NDCG across all K values
fig, ax = plt.subplots(figsize=(8, 4))

for model_name in all_results:
    ndcg_vals = [summary[(summary['model'] == model_name) & (summary['k'] == k)]['ndcg'].values[0] for k in K_VALUES]
    ax.plot(K_VALUES, ndcg_vals, marker='o', label=model_name)

ax.set_title('NDCG across K values')
ax.set_xlabel('K')
ax.set_ylabel('NDCG')
ax.set_xticks(K_VALUES)
ax.legend()
plt.tight_layout()
plt.show()

## 8. Inspect Recommendations for a Specific User

In [ ]:
# Change this to any user_id from the dataset
USER_ID = 'user_00001'
N_RECS  = 10

if USER_ID not in train_loader.user_id_to_idx:
    print(f'User {USER_ID!r} not in training set.')
else:
    user_idx = train_loader.user_id_to_idx[USER_ID]

    # Show user history
    history_ids = loader.get_user_history(USER_ID)
    history_df  = products[products['product_id'].isin(history_ids)][['product_id', 'name', 'category', 'brand', 'price']]
    print(f'--- {USER_ID} purchase/interaction history ({len(history_ids)} products) ---')
    display(history_df.head(10))

    # Recommendations from each model
    all_recs = {}
    for name, model in models.items():
        recs = model.recommend(user_idx, n=N_RECS, exclude_seen=True)
        rec_ids = [train_loader.idx_to_product_id[idx] for idx, _ in recs]
        scores  = [score for _, score in recs]
        rec_df  = products[products['product_id'].isin(rec_ids)][['product_id', 'name', 'category', 'brand', 'price']].copy()
        score_map = dict(zip(rec_ids, scores))
        rec_df['score'] = rec_df['product_id'].map(score_map)
        rec_df = rec_df.set_index('product_id').loc[rec_ids].reset_index()  # preserve rank order
        all_recs[name] = rec_df
        print(f'\n--- {name} recommendations ---')
        display(rec_df)

## 9. Similar Items

In [ ]:
# Change this to any product_id
PRODUCT_ID = 'prod_00001'
N_SIMILAR  = 10

if PRODUCT_ID not in train_loader.product_id_to_idx:
    print(f'Product {PRODUCT_ID!r} not in training set.')
else:
    item_idx = train_loader.product_id_to_idx[PRODUCT_ID]
    seed_product = products[products['product_id'] == PRODUCT_ID][['product_id', 'name', 'category', 'brand', 'price']]
    print(f'Seed product:')
    display(seed_product)

    for name, model in models.items():
        if not hasattr(model, 'get_similar_items'):
            continue
        similar = model.get_similar_items(item_idx, n=N_SIMILAR)
        sim_ids = [train_loader.idx_to_product_id[idx] for idx, _ in similar]
        scores  = [s for _, s in similar]
        sim_df  = products[products['product_id'].isin(sim_ids)][['product_id', 'name', 'category', 'brand', 'price']].copy()
        sim_df['similarity'] = sim_df['product_id'].map(dict(zip(sim_ids, scores)))
        sim_df = sim_df.set_index('product_id').loc[sim_ids].reset_index()
        print(f'\n--- {name} similar items ---')
        display(sim_df)

## 10. Top Popular Products

In [ ]:
top_n = 20
popular = interactions.groupby('product_id').size().sort_values(ascending=False).head(top_n)
popular_df = popular.reset_index()
popular_df.columns = ['product_id', 'interaction_count']
popular_df = popular_df.merge(products[['product_id', 'name', 'category', 'brand', 'price']], on='product_id')

fig, ax = plt.subplots(figsize=(12, 5))
bars = ax.barh(popular_df['name'].str[:40], popular_df['interaction_count'], color='steelblue')
ax.invert_yaxis()
ax.set_title(f'Top {top_n} Products by Interaction Count')
ax.set_xlabel('Interactions')
plt.tight_layout()
plt.show()

display(popular_df)